# Projeto 4 — Clipping comparativo (peso 2)

**Pergunta:** o que este recorte de notícias da Agência Brasil e da CNN Brasil mostra sobre a cobertura de Lula e Flávio Bolsonaro? Transforme textos em dados auditáveis e escreva uma conclusão proporcional ao tamanho da amostra. A base tem 40 matérias selecionadas, dez por combinação de veículo e foco, publicadas em outubro de 2026. **Não é uma amostra aleatória de toda a cobertura.**

Entregue uma pasta `23-projeto-4-noticias/` no seu repositório privado com notebook executado, `README.md` e tabelas de resultado pequenas. O projeto integra Aulas 21–23 e vale peso 2. Crie sua **própria chave Groq**; nunca a salve nem a envie.

## 0. Carregar e conferir a base

Abra no Colab. Se este notebook estiver numa pasta isolada, ele baixa a tabela publicada no repositório da disciplina. Confira 40 linhas, datas, URLs e o equilíbrio 10×4.

In [1]:
import pandas as pd
from pathlib import Path
arquivo_local = Path("dados/noticias_lula_flavio_2026.csv")
if not arquivo_local.exists():
    arquivo_local = Path("../dados/noticias_lula_flavio_2026.csv")
url_base = "https://raw.githubusercontent.com/mateuspestana/extracao_analise_2026/main/aulas/23-extracao-busca-semantica/dados/noticias_lula_flavio_2026.csv"
politica = pd.read_csv(arquivo_local if arquivo_local.exists() else url_base)
print(politica.shape)
print(pd.crosstab(politica["veiculo"], politica["foco_amostral"]))
assert len(politica) == 40

(40, 8)
foco_amostral   Flávio Bolsonaro  Lula
veiculo                               
Agência Brasil                10    10
CNN Brasil                    10    10


## 1. Preparação textual — Aula 21

Preserve `texto_analise`; crie `texto_preparado` com regras justificadas. Conte documentos vazios, duplicatas, tokens antes/depois. Mostre três casos lado a lado; um deve conter nome próprio ou negação.

In [2]:
import re
import unicodedata
from IPython.display import display

# preserva caixa alta, acentos, nomes e negações.
# não removemos stopwords
# operações podem eliminar negações ou alterar nomes e o sentido do texto.
texto_original = politica["texto_analise"].copy(deep=True)

def preparar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = unicodedata.normalize("NFC", str(texto))
    # unifica espaços, quebras de linha e tabulações, sem apagar pontuação
    return re.sub(r"\s+", " ", texto).strip()

def tokens(texto):
    # mantém palavras com hífen/apóstrofo como um token
    return re.findall(r"[^\W_]+(?:[-’'][^\W_]+)*", texto, flags=re.UNICODE)

politica["texto_preparado"] = politica["texto_analise"].map(preparar_texto)
antes = politica["texto_analise"].fillna("").astype(str)
depois = politica["texto_preparado"]
politica["tokens_antes"] = antes.map(lambda t: len(tokens(t)))
politica["tokens_depois"] = depois.map(lambda t: len(tokens(t)))

# Duplicatas = ocorrências além da primeira, considerando apenas textos não vazios
# textos iguais não são excluídos
def contar_duplicatas(serie):
    nao_vazios = serie[serie.str.strip().ne("")]
    return int(nao_vazios.duplicated().sum())

resumo_preparacao = pd.DataFrame({
    "medida": ["Documentos", "Valores ausentes", "Documentos vazios (inclui ausentes)",
               "Textos duplicados além da primeira ocorrência", "Total de tokens"],
    "antes": [len(antes), int(politica["texto_analise"].isna().sum()),
              int(antes.str.strip().eq("").sum()), contar_duplicatas(antes),
              int(politica["tokens_antes"].sum())],
    "depois": [len(depois), int(depois.isna().sum()),
               int(depois.eq("").sum()), contar_duplicatas(depois),
               int(politica["tokens_depois"].sum())]
})
display(resumo_preparacao)

casos_especiais = depois.str.contains(
    r"\b(?:Lula|Flávio|Bolsonaro|não|nunca|nem)\b", case=False, regex=True
)
posicoes = [i for i, valor in enumerate(casos_especiais) if valor][:1]
posicoes += [i for i in range(len(politica)) if i not in posicoes][:3-len(posicoes)]
exemplos_preparacao = politica.iloc[posicoes][[
    "id", "texto_analise", "texto_preparado", "tokens_antes", "tokens_depois"
]].copy()
with pd.option_context("display.max_colwidth", None):
    display(exemplos_preparacao)

assert politica["texto_analise"].equals(texto_original), "O texto original foi alterado."
assert len(politica) == 40
assert politica["texto_preparado"].map(preparar_texto).equals(depois)
print("Documentos com texto alterado:", int(antes.ne(depois).sum()))
print("A limpeza é conservadora; contagens iguais são esperadas quando a base já está limpa.")

,medida,antes,depois
0,Documentos,40,40
1,Valores ausentes,0,0
2,Documentos vazios (inclui ausentes),0,0
3,Textos duplicados além da primeira ocorrência,0,0
4,Total de tokens,1282,1282


,id,texto_analise,texto_preparado,tokens_antes,tokens_depois
0,P01,"Lula defende respeito às religiões no Dia de Nossa Senhora Aparecida. Ao citar o Minha Casa, Minha Vida, o presidente destacou que o tema da campanha da fraternidade da igreja Católica este ano é Fraternidade e Moradia.","Lula defende respeito às religiões no Dia de Nossa Senhora Aparecida. Ao citar o Minha Casa, Minha Vida, o presidente destacou que o tema da campanha da fraternidade da igreja Católica este ano é Fraternidade e Moradia.",37,37
1,P02,"Ato pela reeleição de Lula leva milhares à Avenida Paulista. Candidato à reeleição disse que a semana será decisiva, por causa da votação do fim da escala 6x1, agendada para a próxima quarta-feira (14).","Ato pela reeleição de Lula leva milhares à Avenida Paulista. Candidato à reeleição disse que a semana será decisiva, por causa da votação do fim da escala 6x1, agendada para a próxima quarta-feira (14).",34,34
2,P03,"Meta alega ""problemas técnicos"" para menor alcance de Lula nas redes. O uso ou manipulação de plataformas digitais para direcionar conteúdos e informações durante processos eleitorais tem sido alvo de processos judiciais e debates em todo o mundo.","Meta alega ""problemas técnicos"" para menor alcance de Lula nas redes. O uso ou manipulação de plataformas digitais para direcionar conteúdos e informações durante processos eleitorais tem sido alvo de processos judiciais e debates em todo o mundo.",38,38


Documentos com texto alterado: 0
A limpeza é conservadora; contagens iguais são esperadas quando a base já está limpa.


## 2. Tópicos — Aula 22

Aplique TF-IDF e NMF às 40 linhas. Use poucos tópicos e `random_state=42`. Mostre palavras e manchetes representativas, dê nomes provisórios e faça uma tabela por veículo e foco. Conte também as menções a Lula e Flávio nos títulos e resumos, sempre com o denominador de cada grupo. Explique ao menos um tópico confuso. As editorias do corpus geral da aula não existem nesta base.

In [10]:
%pip install -U scikit-learn

   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/8.4 MB 5.6 MB/s eta 0:00:02
   ------------ --------------------------- 2.6/8.4 MB 6.9 MB/s eta 0:00:01
   -------------------- ------------------- 4.2/8.4 MB 7.1 MB/s eta 0:00:01
   --------------------------- ------------ 5.8/8.4 MB 7.2 MB/s eta 0:00:01
   ----------------------------------- ---- 7.3/8.4 MB 7.1 MB/s eta 0:00:01
   ---------------------------------------- 8.4/8.4 MB 6.9 MB/s  0:00:01
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   - -------------------------------------- 1.6/37.4 MB 8.7 MB/s eta 0:00:05
   --- ------------------------------------ 3.4/37.4 MB 7.8 MB/s eta 0:00:05
   ------ --------------------------------- 5.8/37.4 MB 8.8 MB/s eta 0:00:04
   -------- ------------------------------- 7.9/37.4 MB 9.1 MB/s eta 0:00:04
   ---------- ----------------------------- 10.2/37.4 MB 9.5 MB/s eta 0:00:03
   ------------- -----


[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: C:\Users\mathe\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [12]:
import numpy as np
import pandas as pd
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer


# 1. TF-IDF e NMF


# quatro tópicos mantêm a análise legível para apenas 40 notícias
n_topicos = 4

vectorizador = TfidfVectorizer(
    min_df=2,
    max_df=0.90,
    ngram_range=(1, 2),
    sublinear_tf=True,
    max_features=1000
)

matriz_tfidf = vectorizador.fit_transform(
    politica["texto_preparado"].fillna("")
)

modelo_nmf = NMF(
    n_components=n_topicos,
    init="nndsvda",
    random_state=42,
    max_iter=1000
)

pesos_documentos = modelo_nmf.fit_transform(matriz_tfidf)
termos = vectorizador.get_feature_names_out()


# 2. Palavras principais e nomes provisórios


termos_por_topico = {}
nomes_topicos = {}

for numero in range(n_topicos):
    indices = modelo_nmf.components_[numero].argsort()[::-1][:10]
    termos_topico = termos[indices].tolist()

    termos_por_topico[numero] = termos_topico

    # o nome provisório usa os três termos de maior peso
    nomes_topicos[numero] = (
        f"T{numero + 1} — {' / '.join(termos_topico[:3])}"
    )

tabela_termos = pd.DataFrame({
    "topico_provisorio": [
        nomes_topicos[i] for i in range(n_topicos)
    ],
    "termos_principais": [
        ", ".join(termos_por_topico[i])
        for i in range(n_topicos)
    ]
})

display(tabela_termos)



# 3. Manchetes representativas


representativas = []

for numero in range(n_topicos):
    indices_documentos = (
        pesos_documentos[:, numero]
        .argsort()[::-1][:3]
    )

    for posicao, indice_linha in enumerate(
        indices_documentos, start=1
    ):
        linha = politica.iloc[indice_linha]

        representativas.append({
            "topico_provisorio": nomes_topicos[numero],
            "posicao": posicao,
            "id": linha["id"],
            "manchete": linha["titulo"],
            "peso_nmf": round(
                float(pesos_documentos[indice_linha, numero]),
                3
            )
        })

manchetes_representativas = pd.DataFrame(representativas)
display(manchetes_representativas)



# 4. Tópico dominante de cada notícia

topico_dominante = pesos_documentos.argmax(axis=1)

politica["topico_nmf"] = [
    nomes_topicos[numero]
    for numero in topico_dominante
]


# 5. Tabela de tópicos por veículo e foco


grupos = ["veiculo", "foco_amostral"]

contagem_topicos = pd.crosstab(
    index=[
        politica["veiculo"],
        politica["foco_amostral"]
    ],
    columns=politica["topico_nmf"]
)

denominadores = (
    politica.groupby(grupos)
    .size()
    .reindex(contagem_topicos.index)
)

tabela_topicos_grupo = contagem_topicos.astype(object)

for coluna in tabela_topicos_grupo.columns:
    tabela_topicos_grupo[coluna] = [
        f"{int(valor)}/{int(denominador)} "
        f"({valor / denominador:.0%})"
        for valor, denominador in zip(
            contagem_topicos[coluna],
            denominadores
        )
    ]

tabela_topicos_grupo.insert(
    0,
    "denominador",
    denominadores.astype(int)
)

display(tabela_topicos_grupo.reset_index())



# 6. Menções ao Lula e ao Flávio nos títulos e resumos


padroes = {
    "lula": r"\bLula\b",
    "flavio": r"\bFl[aá]vio(?:\s+Bolsonaro)?\b"
}

for campo in ["titulo", "resumo_fonte"]:
    textos = politica[campo].fillna("").astype(str)

    for pessoa, padrao in padroes.items():
        politica[f"{campo}_menciona_{pessoa}"] = (
            textos.str.contains(
                padrao,
                case=False,
                regex=True,
                na=False
            )
        )

mencoes = (
    politica.groupby(grupos)
    .agg(
        denominador=("id", "size"),
        titulo_lula=("titulo_menciona_lula", "sum"),
        resumo_lula=("resumo_fonte_menciona_lula", "sum"),
        titulo_flavio=("titulo_menciona_flavio", "sum"),
        resumo_flavio=("resumo_fonte_menciona_flavio", "sum")
    )
    .reset_index()
)

tabela_mencoes = mencoes.copy()

colunas_mencoes = [
    "titulo_lula",
    "resumo_lula",
    "titulo_flavio",
    "resumo_flavio"
]

for coluna in colunas_mencoes:
    tabela_mencoes[coluna] = [
        f"{int(valor)}/{int(denominador)} "
        f"({valor / denominador:.0%})"
        for valor, denominador in zip(
            mencoes[coluna],
            mencoes["denominador"]
        )
    ]

display(tabela_mencoes)


# 7. Leitura crítica de um tópico confuso


pesos_ordenados = np.sort(pesos_documentos, axis=1)

politica["margem_topico"] = (
    pesos_ordenados[:, -1] -
    pesos_ordenados[:, -2]
)

margens_medias = (
    politica.groupby("topico_nmf")["margem_topico"]
    .mean()
)

topico_confuso = margens_medias.idxmin()

indice_confuso = next(
    numero
    for numero, nome in nomes_topicos.items()
    if nome == topico_confuso
)

distribuicao_confusa = (
    politica.loc[
        politica["topico_nmf"].eq(topico_confuso),
        grupos
    ]
    .value_counts()
    .rename("n")
    .reset_index()
)

print("LEITURA CRÍTICA")
print(
    f"{topico_confuso} foi o tópico mais ambíguo. "
    f"A margem média entre os dois maiores pesos foi "
    f"{margens_medias.loc[topico_confuso]:.3f}. "
    f"Seus termos principais foram "
    f"{', '.join(termos_por_topico[indice_confuso][:5])}."
)

display(distribuicao_confusa)

print(
    "Esse tópico pode ser considerado confuso porque uma mesma "
    "notícia pode mencionar candidatos, campanha, apoios e "
    "instituições eleitorais. O NMF identifica coocorrências de "
    "palavras, não editorias fixas. Portanto, os resultados "
    "descrevem apenas este recorte de 40 notícias e não comprovam "
    "viés de qualquer veículo."
)

,topico_provisorio,termos_principais
0,T1 — de / em / que,"de, em, que, flávio, da, bolsonaro, lula, fláv..."
1,T2 — canal / tse / exército,"canal, tse, exército, exército do, do lula, ca..."
2,T3 — turno / segundo turno / segundo,"turno, segundo turno, segundo, senador, presid..."
3,T4 — por / de / site,"por, de, site, oferece, de lula, site oferece,..."


,topico_provisorio,posicao,id,manchete,peso_nmf
0,T1 — de / em / que,1,P15,Flávio Bolsonaro faz ato no Rio ao lado de Dou...,0.342
1,T1 — de / em / que,2,P04,Estudantes fazem ato no Rio de Janeiro em defe...,0.324
2,T1 — de / em / que,3,P11,Flávio passa domingo reunido com prefeitos e v...,0.321
3,T2 — canal / tse / exército,1,P27,"Canal ""Exército do Lula"" é registrado junto ao...",0.668
4,T2 — canal / tse / exército,2,P05,Técnicos do TSE não encontram conteúdo falso n...,0.665
5,T2 — canal / tse / exército,3,P09,Campanha registra canal Exército do Lula na Ju...,0.598
6,T3 — turno / segundo turno / segundo,1,P39,Flávio: Bolsonaro está satisfeito com 1º turno...,0.706
7,T3 — turno / segundo turno / segundo,2,P38,Flávio buscará orientação de Bolsonaro para re...,0.705
8,T3 — turno / segundo turno / segundo,3,P37,Centrão mira pastas com maior orçamento se Flá...,0.482
9,T4 — por / de / site,1,P16,Site oferece criptomoedas por posts de apoio a...,0.595


topico_nmf,veiculo,foco_amostral,denominador,T1 — de / em / que,T2 — canal / tse / exército,T3 — turno / segundo turno / segundo,T4 — por / de / site
0,Agência Brasil,Flávio Bolsonaro,10,6/10 (60%),0/10 (0%),1/10 (10%),3/10 (30%)
1,Agência Brasil,Lula,10,6/10 (60%),3/10 (30%),0/10 (0%),1/10 (10%)
2,CNN Brasil,Flávio Bolsonaro,10,4/10 (40%),1/10 (10%),4/10 (40%),1/10 (10%)
3,CNN Brasil,Lula,10,6/10 (60%),2/10 (20%),0/10 (0%),2/10 (20%)


,veiculo,foco_amostral,denominador,titulo_lula,resumo_lula,titulo_flavio,resumo_flavio
0,Agência Brasil,Flávio Bolsonaro,10,2/10 (20%),4/10 (40%),10/10 (100%),3/10 (30%)
1,Agência Brasil,Lula,10,10/10 (100%),3/10 (30%),1/10 (10%),3/10 (30%)
2,CNN Brasil,Flávio Bolsonaro,10,1/10 (10%),5/10 (50%),10/10 (100%),1/10 (10%)
3,CNN Brasil,Lula,10,10/10 (100%),0/10 (0%),1/10 (10%),1/10 (10%)


LEITURA CRÍTICA
T1 — de / em / que foi o tópico mais ambíguo. A margem média entre os dois maiores pesos foi 0.165. Seus termos principais foram de, em, que, flávio, da.


,veiculo,foco_amostral,n
0,Agência Brasil,Lula,6
1,Agência Brasil,Flávio Bolsonaro,6
2,CNN Brasil,Lula,6
3,CNN Brasil,Flávio Bolsonaro,4


Esse tópico pode ser considerado confuso porque uma mesma notícia pode mencionar candidatos, campanha, apoios e instituições eleitorais. O NMF identifica coocorrências de palavras, não editorias fixas. Portanto, os resultados descrevem apenas este recorte de 40 notícias e não comprovam viés de qualquer veículo.


## 3. Extração estruturada — Aula 23

Crie sua própria chave Groq. Reaproveite o esquema e a função `extrair()` da aula principal. Selecione **4 notícias por combinação de veículo e foco** com `groupby(...).sample(n=4, random_state=42)`; são 16 chamadas ao todo. Faça primeiro **uma** chamada de teste. Depois rode o lote **uma vez** e guarde a tabela. Não repita o lote ao ajustar gráficos ou texto do relatório.

Confira se `evidencia` é um trecho literal de `texto_analise`. Audite manualmente pelo menos quatro fichas, uma por combinação. Marque ausências, exageros e atribuições ambíguas; não conserte silenciosamente a saída do modelo.

In [13]:
from getpass import getpass
chave_groq = getpass("Sua chave Groq (não aparece): ")
amostra_ia = politica.groupby(["veiculo", "foco_amostral"], group_keys=False).sample(n=4, random_state=42)
print(pd.crosstab(amostra_ia.veiculo, amostra_ia.foco_amostral))


foco_amostral   Flávio Bolsonaro  Lula
veiculo                               
Agência Brasil                 4     4
CNN Brasil                     4     4


In [14]:
import requests, json
modelo_groq = "openai/gpt-oss-20b"
esquema = {
    "type": "object", "additionalProperties": False,
    "properties": {
        "pessoa_central": {"type": ["string", "null"]},
        "fato": {"type": "string"},
        "tema": {"type": "string"},
        "evidencia": {"type": "string"},
        "limite": {"type": "string"}
    },
    "required": ["pessoa_central", "fato", "tema", "evidencia", "limite"]
}

def extrair(texto):
    resposta = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {chave_groq}"},
        json={"model": modelo_groq, "temperature": 0, "reasoning_effort": "low", "max_completion_tokens": 600,
              "messages": [{"role": "system", "content": "Extraia apenas fatos presentes no texto recebido. A evidencia deve ser um trecho literal e contínuo do texto. Se não houver pessoa central, use null. Não infira intenção política nem viés do veículo."},
                           {"role": "user", "content": texto}],
              "response_format": {"type": "json_schema", "json_schema": {"name": "ficha_noticia", "strict": True, "schema": esquema}}},
        timeout=40)
    if resposta.status_code == 429:
        raise RuntimeError("Limite Groq atingido (429). Aguarde antes de tentar outra chamada.")
    resposta.raise_for_status()
    return json.loads(resposta.json()["choices"][0]["message"]["content"])


In [15]:
# Teste uma notícia antes de rodar o lote.
teste = extrair(amostra_ia.iloc[0]["texto_analise"])
print(teste)
print("Evidência literal:", teste["evidencia"] in amostra_ia.iloc[0]["texto_analise"])


{'pessoa_central': 'Flávio', 'fato': 'TSE manda suspender site que oferece criptomoedas por apoio a Flávio', 'tema': 'eleições', 'evidencia': 'TSE manda suspender site que oferece criptomoedas por apoio a Flávio', 'limite': 'null'}
Evidência literal: True


In [22]:
 fichas = []
 for _, linha in amostra_ia.iterrows():
     resultado = extrair(linha["texto_analise"])
     fichas.append({"id": linha["id"], "veiculo": linha["veiculo"],
                    "foco_amostral": linha["foco_amostral"], **resultado,
                    "evidencia_literal": resultado["evidencia"] in linha["texto_analise"]})
 fichas = pd.DataFrame(fichas)
 fichas.to_csv("fichas_extraidas.csv", index=False)

## 4. Busca — Aula 23

Faça uma pergunta substantiva. Compare top 3 de busca por palavra e de busca semântica nas 40 matérias. Registre IDs, títulos, links e uma avaliação manual da relevância. A consulta precisa poder ser repetida.

In [ ]:
import re
import pandas as pd

# Pergunta: Quantas vezes Lula e Flávio Bolsonaro foram citados nos títulos e resumos das 40 notícias?

# textos nos quais as menções serão contadas
titulos = politica["titulo"].fillna("").astype(str)
resumos = politica["resumo_fonte"].fillna("").astype(str)

# conta “Lula” e “Flávio” ou “Flávio Bolsonaro”.
# “Bolsonaro” sozinho não é contado, pois poderia se referir a Jair Bolsonaro
padroes = {
    "Lula": r"\bLula\b",
    "Flávio Bolsonaro": r"\bFl[aá]vio(?:\s+Bolsonaro)?\b"
}

resultados = []

for candidato, padrao in padroes.items():
    mencoes_titulos = titulos.str.count(
        padrao,
        flags=re.IGNORECASE
    ).sum()

    mencoes_resumos = resumos.str.count(
        padrao,
        flags=re.IGNORECASE
    ).sum()

    total = mencoes_titulos + mencoes_resumos

    resultados.append({
        "candidato": candidato,
        "menções nos títulos": int(mencoes_titulos),
        "menções nos resumos": int(mencoes_resumos),
        "total de menções": int(total)
    })

tabela_mencoes = pd.DataFrame(resultados)

print(
    "Pergunta: quantas vezes Lula e Flávio Bolsonaro foram "
    "citados nos títulos e resumos das 40 notícias?"
)

display(tabela_mencoes)

Pergunta: quantas vezes Lula e Flávio Bolsonaro foram citados nos títulos e resumos das 40 notícias?


,candidato,menções nos títulos,menções nos resumos,total de menções
0,Lula,23,12,35
1,Flávio Bolsonaro,22,8,30


## 5. Relatório e entrega

No `README.md`, siga esta ordem: objetivo e pergunta; preparação; revisão de 21–22; roteiro de análise; prática aplicada; resultados do desafio; entrega; declaração de uso de IA; checklist. Inclua fonte, janela temporal, regra de seleção, tabela de denominadores, pelo menos dois achados com IDs/URLs, quatro fichas auditadas, tópico confuso, comparação de buscas e limitações. Separe **descrição dos dados** de **interpretação**. Não chame a diferença observada de “viés do jornal” sem um protocolo mais amplo.

Faça um commit de progresso e um de entrega. Prepare-se para explicar uma decisão de limpeza, um tópico, uma ficha e uma busca.

- [ ] 40 linhas: 10 por veículo × foco.
- [ ] Chave pessoal fora dos arquivos entregues.
- [ ] Extração em amostra de 16, com quatro auditorias.
- [ ] Conclusões citam IDs e links e respeitam o recorte.
- [ ] Notebook executa de cima a baixo no Colab.